# Phase 0 Non-Idle Action Diagnostics

This notebook plots the Phase 0 diagnostics used before introducing an intervention gate. It is intentionally smaller than `wandb_run_plots.ipynb`: load cached W&B histories, select the phase-0 metrics, and generate a few repeatable Plotly figures.

Metrics covered:

- `train/non_idle_agents_count_0_frac` to `train/non_idle_agents_count_3_frac`
- `train/frac_any_non_idle`
- `train/frac_multi_agent_non_idle`
- `train/frac_action_0_agent_*`
- `train/illegal_action_rate_agent_*`

In [ ]:
from pathlib import Path
import json
import re
from typing import Iterable

import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError("Install plotly first, for example: pip install plotly") from exc

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
PLOT_TEMPLATE = "plotly_white"


## Configuration

Edit the filters below to focus on a family of runs. Empty `RUN_NAME_INCLUDE` means all cached runs are considered. The loader automatically skips runs that do not contain any Phase 0 metric.

In [ ]:
cwd = Path.cwd().resolve()
if cwd.name == "configs" and cwd.parent.name == "Topology_Task":
    TASK_DIR = cwd.parent
elif (cwd / "Topology_Task").exists():
    TASK_DIR = cwd / "Topology_Task"
else:
    TASK_DIR = cwd

REPO_ROOT = TASK_DIR.parent
CACHE_DIR = TASK_DIR / "outputs" / "wandb_cache"
FULL_HISTORY_DIR = CACHE_DIR / "full_history"
CACHE_INDEX_PATH = CACHE_DIR / "full_history_cache_index.csv"
FIGURE_DIR = TASK_DIR / "outputs" / "wandb_figures" / "phase0_non_idle"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

WANDB_ENTITY = "corentin-plumet-epfl"
WANDB_PROJECT = "Grid2Op"

# Examples: [r"best_00", r"intervention_gate"]. Leave empty to use all cached runs.
RUN_NAME_INCLUDE: list[str] = []
RUN_NAME_EXCLUDE: list[str] = []
MAX_RUNS: int | None = None
SMOOTH_WINDOW = 5

AGENT_IDS = ["agent_0", "agent_1", "agent_2"]
NON_IDLE_COUNT_METRICS = [f"train/non_idle_agents_count_{i}_frac" for i in range(4)]
SUMMARY_METRICS = ["train/frac_any_non_idle", "train/frac_multi_agent_non_idle"]
ACTION0_PATTERN = re.compile(r"^train/frac_action_0_(agent_\d+)$")
ILLEGAL_PATTERN = re.compile(r"^train/illegal_action_rate_(agent_\d+)$")
PHASE0_BASE_METRICS = NON_IDLE_COUNT_METRICS + SUMMARY_METRICS

print(f"Task dir: {TASK_DIR}")
print(f"Cache dir: {CACHE_DIR}")
print(f"Figure dir: {FIGURE_DIR}")


## Cache Discovery And Loading Helpers

In [ ]:
def _is_missing(value) -> bool:
    return value is None or (isinstance(value, float) and np.isnan(value)) or str(value) == "nan" or str(value) == ""


def _as_path(value, base: Path | None = None) -> Path | None:
    if _is_missing(value):
        return None
    path = Path(str(value)).expanduser()
    if not path.is_absolute() and base is not None:
        path = base / path
    return path


def _read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def infer_seed(run_name: str, run_id: str = "") -> int | None:
    text = f"{run_name} {run_id}"
    for pattern in [r"(?:^|_)s(\d+)(?:_|$)", r"_T_(\d+)_"]:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1))
    return None


def infer_family(run_name: str) -> str:
    family = re.sub(r"__MAPPO.*$", "", str(run_name))
    family = re.sub(r"(?:^|_)s\d+(?=_|$)", "", family)
    family = re.sub(r"_job\d+", "", family)
    family = re.sub(r"_+", "_", family).strip("_")
    return family or str(run_name)


def _passes_filters(name: str, include: Iterable[str], exclude: Iterable[str]) -> bool:
    include = list(include or [])
    exclude = list(exclude or [])
    if include and not any(re.search(pattern, name) for pattern in include):
        return False
    if exclude and any(re.search(pattern, name) for pattern in exclude):
        return False
    return True


def _metadata_paths_from_cache() -> list[Path]:
    if not FULL_HISTORY_DIR.exists():
        return []
    return sorted(FULL_HISTORY_DIR.glob("*/metadata.json"))


def _row_from_metadata(meta_path: Path) -> dict:
    meta = _read_json(meta_path)
    parent = meta_path.parent
    folder_name = parent.name
    guessed_name, _, guessed_id = folder_name.partition("__")
    run_name = meta.get("run_name") or meta.get("name") or guessed_name
    run_id = meta.get("run_id") or meta.get("id") or guessed_id
    parquet_path = _as_path(meta.get("history_parquet"), parent) or parent / "history.parquet"
    csv_path = _as_path(meta.get("history_csv"), parent) or parent / "history.csv.gz"
    return {
        "run_name": str(run_name),
        "run_id": str(run_id),
        "run_family": infer_family(str(run_name)),
        "seed": infer_seed(str(run_name), str(run_id)),
        "history_parquet": str(parquet_path) if parquet_path.exists() else None,
        "history_csv": str(csv_path) if csv_path.exists() else None,
        "metadata_path": str(meta_path),
        "cache_dir": str(parent),
    }


def discover_cached_runs() -> pd.DataFrame:
    rows: list[dict] = []

    if CACHE_INDEX_PATH.exists():
        index = pd.read_csv(CACHE_INDEX_PATH)
        for _, raw in index.iterrows():
            raw_dict = raw.to_dict()
            run_name = raw_dict.get("run_name") or raw_dict.get("name") or raw_dict.get("display_name")
            run_id = raw_dict.get("run_id") or raw_dict.get("id") or raw_dict.get("wandb_run_id")
            if _is_missing(run_name):
                continue
            cache_dir = _as_path(raw_dict.get("cache_dir"), CACHE_DIR)
            parquet_path = _as_path(raw_dict.get("history_parquet"), CACHE_DIR)
            csv_path = _as_path(raw_dict.get("history_csv"), CACHE_DIR)
            if cache_dir is not None:
                parquet_path = parquet_path or cache_dir / "history.parquet"
                csv_path = csv_path or cache_dir / "history.csv.gz"
            rows.append({
                "run_name": str(run_name),
                "run_id": "" if _is_missing(run_id) else str(run_id),
                "run_family": infer_family(str(run_name)),
                "seed": infer_seed(str(run_name), "" if _is_missing(run_id) else str(run_id)),
                "history_parquet": str(parquet_path) if parquet_path and parquet_path.exists() else None,
                "history_csv": str(csv_path) if csv_path and csv_path.exists() else None,
                "metadata_path": raw_dict.get("metadata_path"),
                "cache_dir": str(cache_dir) if cache_dir else None,
            })

    rows.extend(_row_from_metadata(path) for path in _metadata_paths_from_cache())
    if not rows:
        return pd.DataFrame(columns=["run_name", "run_id", "run_family", "seed", "history_parquet", "history_csv"])

    runs = pd.DataFrame(rows)
    runs = runs.drop_duplicates(subset=["run_name", "run_id", "history_parquet", "history_csv"], keep="last")
    runs = runs[(runs["history_parquet"].notna()) | (runs["history_csv"].notna())].copy()
    runs = runs.sort_values(["run_family", "seed", "run_name"], na_position="last").reset_index(drop=True)
    return runs


def phase0_metric_columns(columns: Iterable[str]) -> list[str]:
    columns = list(columns)
    exact = [metric for metric in PHASE0_BASE_METRICS if metric in columns]
    action0 = sorted(col for col in columns if ACTION0_PATTERN.match(col))
    illegal = sorted(col for col in columns if ILLEGAL_PATTERN.match(col))
    return exact + action0 + illegal


In [ ]:
def _history_read_columns(wanted_metrics: list[str]) -> list[str]:
    step_candidates = ["_step", "step", "charts/global_step"]
    return list(dict.fromkeys(step_candidates + wanted_metrics))


def read_history_table(row: pd.Series, wanted_metrics: list[str]) -> pd.DataFrame:
    parquet_path = _as_path(row.get("history_parquet"))
    csv_path = _as_path(row.get("history_csv"))
    wanted_columns = _history_read_columns(wanted_metrics)

    if parquet_path and parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path, columns=wanted_columns)
        except Exception:
            if csv_path and csv_path.exists():
                return pd.read_csv(csv_path)
            return pd.read_parquet(parquet_path)

    if csv_path and csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(f"No cached history found for {row.get('run_name')}")


def normalize_history_steps(history: pd.DataFrame) -> pd.DataFrame:
    history = history.copy()
    if "_step" not in history.columns:
        if "step" in history.columns:
            history["_step"] = history["step"]
        elif "charts/global_step" in history.columns:
            history["_step"] = history["charts/global_step"]
        else:
            history["_step"] = np.arange(len(history))
    history["_step"] = pd.to_numeric(history["_step"], errors="coerce")
    return history.dropna(subset=["_step"])


def load_phase0_histories(
    runs: pd.DataFrame,
    include: Iterable[str] = (),
    exclude: Iterable[str] = (),
    max_runs: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    skipped = []
    wanted_metrics = PHASE0_BASE_METRICS + [f"train/frac_action_0_{agent}" for agent in AGENT_IDS] + [f"train/illegal_action_rate_{agent}" for agent in AGENT_IDS]
    selected = runs[runs["run_name"].map(lambda name: _passes_filters(str(name), include, exclude))].copy()
    if max_runs is not None:
        selected = selected.head(max_runs)

    for idx, run in selected.iterrows():
        try:
            history = normalize_history_steps(read_history_table(run, wanted_metrics))
        except Exception as exc:
            skipped.append({"run_name": run.get("run_name"), "reason": repr(exc)})
            continue

        metric_cols = phase0_metric_columns(history.columns)
        if not metric_cols:
            skipped.append({"run_name": run.get("run_name"), "reason": "no phase-0 metrics"})
            continue

        keep_cols = ["_step"] + metric_cols
        if "charts/global_step" in history.columns:
            keep_cols.append("charts/global_step")
        frame = history[keep_cols].copy()
        frame["run_name"] = run["run_name"]
        frame["run_id"] = run.get("run_id", "")
        frame["run_family"] = run.get("run_family", infer_family(run["run_name"]))
        frame["seed"] = run.get("seed", infer_seed(run["run_name"], run.get("run_id", "")))
        rows.append(frame)

    if not rows:
        empty_cols = ["_step", "run_name", "run_id", "run_family", "seed"] + wanted_metrics
        return pd.DataFrame(columns=empty_cols), pd.DataFrame(skipped)

    phase0 = pd.concat(rows, ignore_index=True, sort=False)
    metric_cols = phase0_metric_columns(phase0.columns)
    for col in metric_cols:
        phase0[col] = pd.to_numeric(phase0[col], errors="coerce")
    phase0 = phase0.sort_values(["run_family", "seed", "run_name", "_step"], na_position="last").reset_index(drop=True)
    return phase0, pd.DataFrame(skipped)


def summarize_phase0_runs(history: pd.DataFrame) -> pd.DataFrame:
    if history.empty:
        return pd.DataFrame()
    metric_cols = phase0_metric_columns(history.columns)
    final = history.sort_values("_step").groupby("run_name", as_index=False).tail(1)
    summary_cols = ["run_name", "run_family", "seed", "_step"] + metric_cols
    return final[summary_cols].sort_values(["run_family", "seed", "run_name"], na_position="last")


## Load Phase 0 Histories

In [ ]:
runs_df = discover_cached_runs()
print(f"Cached runs found: {len(runs_df)}")
display(runs_df.head(10))

phase0_history, skipped_runs = load_phase0_histories(
    runs_df,
    include=RUN_NAME_INCLUDE,
    exclude=RUN_NAME_EXCLUDE,
    max_runs=MAX_RUNS,
)

print(f"Runs with Phase 0 metrics: {phase0_history['run_name'].nunique() if not phase0_history.empty else 0}")
print(f"Rows loaded: {len(phase0_history)}")
if not skipped_runs.empty:
    print(f"Skipped runs: {len(skipped_runs)}")
    display(skipped_runs.head(20))

phase0_summary = summarize_phase0_runs(phase0_history)
display(phase0_summary.head(20))


## Plot Helpers

In [ ]:
def _metric_label(metric: str) -> str:
    return (
        metric.replace("train/", "")
        .replace("non_idle_agents_count_", "count=")
        .replace("_frac", "")
        .replace("frac_", "")
        .replace("illegal_action_rate_", "illegal ")
        .replace("action_0_", "action 0 ")
    )


def _smooth_long(df: pd.DataFrame, group_cols: list[str], value_col: str = "value", window: int = SMOOTH_WINDOW) -> pd.DataFrame:
    if window <= 1 or df.empty:
        return df
    out = df.sort_values(group_cols + ["_step"]).copy()
    out[value_col] = out.groupby(group_cols, dropna=False)[value_col].transform(
        lambda values: values.rolling(window=window, min_periods=1).mean()
    )
    return out


def _selected_history(history: pd.DataFrame, include: Iterable[str] = (), exclude: Iterable[str] = ()) -> pd.DataFrame:
    if history.empty:
        return history.copy()
    mask = history["run_name"].map(lambda name: _passes_filters(str(name), include, exclude))
    return history[mask].copy()


def _aggregate_by_step(history: pd.DataFrame, metric_cols: list[str], group_cols: list[str] | None = None) -> pd.DataFrame:
    group_cols = group_cols or []
    cols = ["_step"] + group_cols + metric_cols
    return history[cols].groupby(["_step"] + group_cols, dropna=False, as_index=False)[metric_cols].mean()


def save_figure(fig, name: str, output_dir: Path = FIGURE_DIR) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print(f"Saved {path}")
    return path


In [ ]:
def plot_non_idle_distribution(
    history: pd.DataFrame,
    include: Iterable[str] = (),
    exclude: Iterable[str] = (),
    aggregate: bool = True,
):
    data = _selected_history(history, include, exclude)
    count_cols = [col for col in NON_IDLE_COUNT_METRICS if col in data.columns]
    if data.empty or not count_cols:
        raise ValueError("No non-idle count metrics available for the selected runs.")

    if aggregate:
        data = _aggregate_by_step(data, count_cols)
        title_suffix = "mean across selected runs"
    else:
        title_suffix = "per run"

    long = data.melt(
        id_vars=[col for col in ["_step", "run_name", "run_family", "seed"] if col in data.columns],
        value_vars=count_cols,
        var_name="metric",
        value_name="fraction",
    ).dropna(subset=["fraction"])
    long["non_idle_agents"] = long["metric"].str.extract(r"count_(\d+)_frac").astype(int)

    fig = go.Figure()
    for count, part in long.groupby("non_idle_agents"):
        part = part.sort_values("_step")
        fig.add_trace(
            go.Scatter(
                x=part["_step"],
                y=part["fraction"],
                mode="lines",
                stackgroup="one" if aggregate else None,
                name=f"{count} non-idle agents",
                hovertemplate="step=%{x}<br>fraction=%{y:.3f}<extra></extra>",
            )
        )

    fig.update_layout(
        title=f"Distribution of non-idle agents per env step ({title_suffix})",
        xaxis_title="training step",
        yaxis_title="fraction of rollout steps",
        yaxis=dict(range=[0, 1]),
        template=PLOT_TEMPLATE,
        height=520,
        legend_title_text="joint action sparsity",
    )
    return fig


def plot_non_idle_summary(history: pd.DataFrame, include: Iterable[str] = (), exclude: Iterable[str] = (), aggregate: bool = False):
    data = _selected_history(history, include, exclude)
    metrics = [metric for metric in SUMMARY_METRICS if metric in data.columns]
    if data.empty or not metrics:
        raise ValueError("No summary non-idle metrics available for the selected runs.")

    if aggregate:
        data = _aggregate_by_step(data, metrics, group_cols=["run_family"] if "run_family" in data.columns else [])

    id_cols = [col for col in ["_step", "run_name", "run_family", "seed"] if col in data.columns]
    long = data.melt(id_vars=id_cols, value_vars=metrics, var_name="metric", value_name="value").dropna(subset=["value"])
    group_cols = [col for col in ["run_name", "run_family", "metric"] if col in long.columns]
    long = _smooth_long(long, group_cols=group_cols)
    long["metric_label"] = long["metric"].map(_metric_label)

    color = "metric_label" if aggregate else "run_family"
    line_dash = None if aggregate else "metric_label"
    fig = px.line(
        long,
        x="_step",
        y="value",
        color=color,
        line_dash=line_dash,
        hover_data=[col for col in ["run_name", "seed", "metric_label"] if col in long.columns],
        title="Any intervention and multi-agent intervention frequency",
        template=PLOT_TEMPLATE,
        labels={"_step": "training step", "value": "fraction", "metric_label": "metric", "run_family": "run family"},
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=560)
    return fig


In [ ]:
def _agent_metric_long(history: pd.DataFrame, pattern: re.Pattern, value_name: str, include: Iterable[str] = (), exclude: Iterable[str] = ()) -> pd.DataFrame:
    data = _selected_history(history, include, exclude)
    metric_cols = [col for col in data.columns if pattern.match(col)]
    if data.empty or not metric_cols:
        return pd.DataFrame()
    id_cols = [col for col in ["_step", "run_name", "run_family", "seed"] if col in data.columns]
    long = data.melt(id_vars=id_cols, value_vars=metric_cols, var_name="metric", value_name=value_name)
    long = long.dropna(subset=[value_name])
    long["agent"] = long["metric"].map(lambda metric: pattern.match(metric).group(1))
    return long


def plot_action0_by_agent(history: pd.DataFrame, include: Iterable[str] = (), exclude: Iterable[str] = (), aggregate: bool = False):
    long = _agent_metric_long(history, ACTION0_PATTERN, "fraction_action0", include, exclude)
    if long.empty:
        raise ValueError("No per-agent action-0 metrics available for the selected runs.")

    if aggregate:
        long = long.groupby(["_step", "agent"], as_index=False)["fraction_action0"].mean()
        color = "agent"
        line_dash = None
    else:
        long = _smooth_long(long, group_cols=["run_name", "agent"], value_col="fraction_action0")
        color = "agent"
        line_dash = "run_family" if "run_family" in long.columns else None

    fig = px.line(
        long,
        x="_step",
        y="fraction_action0",
        color=color,
        line_dash=line_dash,
        hover_data=[col for col in ["run_name", "run_family", "seed"] if col in long.columns],
        title="Fraction of action 0 by agent",
        template=PLOT_TEMPLATE,
        labels={"_step": "training step", "fraction_action0": "fraction action 0", "agent": "agent"},
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=560)
    return fig


def plot_illegal_rate_by_agent(history: pd.DataFrame, include: Iterable[str] = (), exclude: Iterable[str] = (), aggregate: bool = False):
    long = _agent_metric_long(history, ILLEGAL_PATTERN, "illegal_rate", include, exclude)
    if long.empty:
        raise ValueError("No per-agent illegal-action metrics available for the selected runs.")

    if aggregate:
        long = long.groupby(["_step", "agent"], as_index=False)["illegal_rate"].mean()
        line_dash = None
    else:
        long = _smooth_long(long, group_cols=["run_name", "agent"], value_col="illegal_rate")
        line_dash = "run_family" if "run_family" in long.columns else None

    fig = px.line(
        long,
        x="_step",
        y="illegal_rate",
        color="agent",
        line_dash=line_dash,
        hover_data=[col for col in ["run_name", "run_family", "seed"] if col in long.columns],
        title="Illegal action rate by agent",
        template=PLOT_TEMPLATE,
        labels={"_step": "training step", "illegal_rate": "illegal action rate", "agent": "agent"},
    )
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(height=560)
    return fig


In [ ]:
def plot_final_snapshot(history: pd.DataFrame, include: Iterable[str] = (), exclude: Iterable[str] = ()):
    data = _selected_history(history, include, exclude)
    if data.empty:
        raise ValueError("No data for selected runs.")
    final = data.sort_values("_step").groupby("run_name", as_index=False).tail(1).copy()
    final["label"] = final["run_name"].str.replace("__MAPPO.*$", "", regex=True)

    count_cols = [col for col in NON_IDLE_COUNT_METRICS if col in final.columns]
    action0_cols = [col for col in final.columns if ACTION0_PATTERN.match(col)]
    illegal_cols = [col for col in final.columns if ILLEGAL_PATTERN.match(col)]

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Final non-idle count distribution",
            "Final any/multi-agent intervention",
            "Final action 0 fraction by agent",
            "Final illegal action rate by agent",
        ),
        horizontal_spacing=0.10,
        vertical_spacing=0.14,
    )

    for col in count_cols:
        count = re.search(r"count_(\d+)_frac", col).group(1)
        fig.add_trace(go.Bar(x=final["label"], y=final[col], name=f"{count} non-idle"), row=1, col=1)

    for col in [metric for metric in SUMMARY_METRICS if metric in final.columns]:
        fig.add_trace(go.Bar(x=final["label"], y=final[col], name=_metric_label(col)), row=1, col=2)

    for col in action0_cols:
        agent = ACTION0_PATTERN.match(col).group(1)
        fig.add_trace(go.Bar(x=final["label"], y=final[col], name=f"action 0 {agent}"), row=2, col=1)

    for col in illegal_cols:
        agent = ILLEGAL_PATTERN.match(col).group(1)
        fig.add_trace(go.Bar(x=final["label"], y=final[col], name=f"illegal {agent}"), row=2, col=2)

    fig.update_layout(
        title="Phase 0 final-step snapshot",
        template=PLOT_TEMPLATE,
        height=900,
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=-0.24, xanchor="left", x=0),
    )
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=2)
    fig.update_yaxes(range=[0, 1], row=2, col=1)
    fig.update_xaxes(tickangle=35)
    return fig


## Make The Phase 0 Plots

Use `PLOT_INCLUDE` and `PLOT_EXCLUDE` to focus on a subset after loading. For example: `PLOT_INCLUDE = [r"best_00"]`.

In [ ]:
PLOT_INCLUDE: list[str] = RUN_NAME_INCLUDE
PLOT_EXCLUDE: list[str] = RUN_NAME_EXCLUDE

fig_distribution = plot_non_idle_distribution(phase0_history, PLOT_INCLUDE, PLOT_EXCLUDE, aggregate=True)
fig_distribution.show()

fig_summary = plot_non_idle_summary(phase0_history, PLOT_INCLUDE, PLOT_EXCLUDE, aggregate=False)
fig_summary.show()

fig_action0 = plot_action0_by_agent(phase0_history, PLOT_INCLUDE, PLOT_EXCLUDE, aggregate=False)
fig_action0.show()

fig_illegal = plot_illegal_rate_by_agent(phase0_history, PLOT_INCLUDE, PLOT_EXCLUDE, aggregate=False)
fig_illegal.show()

fig_snapshot = plot_final_snapshot(phase0_history, PLOT_INCLUDE, PLOT_EXCLUDE)
fig_snapshot.show()


## Save Figures

In [ ]:
SAVE_PLOTS = False

if SAVE_PLOTS:
    save_figure(fig_distribution, "phase0_non_idle_distribution")
    save_figure(fig_summary, "phase0_any_multi_non_idle")
    save_figure(fig_action0, "phase0_action0_by_agent")
    save_figure(fig_illegal, "phase0_illegal_rate_by_agent")
    save_figure(fig_snapshot, "phase0_final_snapshot")


## Optional: Recover A Run With `scan_history()`

Use this only when the local full-history cache does not contain a run yet. Put W&B run ids or full run refs in `WANDB_RUN_REFS`, then run the cell. The output is a dataframe that you can concatenate with `phase0_history`.

In [ ]:
WANDB_RUN_REFS: list[str] = []


def _normal_wandb_ref(run_ref: str) -> str:
    if run_ref.count("/") == 2:
        return run_ref
    return f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run_ref}"


def fetch_phase0_with_scan_history(run_refs: list[str], page_size: int = 1000) -> pd.DataFrame:
    if not run_refs:
        return pd.DataFrame()
    try:
        import wandb
    except ImportError as exc:
        raise ImportError("Install wandb first, for example: pip install wandb") from exc

    api = wandb.Api()
    keys = _history_read_columns(PHASE0_BASE_METRICS + [f"train/frac_action_0_{agent}" for agent in AGENT_IDS] + [f"train/illegal_action_rate_{agent}" for agent in AGENT_IDS])
    histories = []
    for run_ref in run_refs:
        run = api.run(_normal_wandb_ref(run_ref))
        rows = list(run.scan_history(keys=keys, page_size=page_size))
        if not rows:
            print(f"No rows returned for {run_ref}")
            continue
        history = normalize_history_steps(pd.DataFrame(rows))
        metric_cols = phase0_metric_columns(history.columns)
        if not metric_cols:
            print(f"No phase-0 metrics found for {run_ref}")
            continue
        history = history[["_step"] + metric_cols].copy()
        history["run_name"] = run.name
        history["run_id"] = run.id
        history["run_family"] = infer_family(run.name)
        history["seed"] = infer_seed(run.name, run.id)
        histories.append(history)
    return pd.concat(histories, ignore_index=True, sort=False) if histories else pd.DataFrame()


scan_history_phase0 = fetch_phase0_with_scan_history(WANDB_RUN_REFS)
if not scan_history_phase0.empty:
    phase0_history = pd.concat([phase0_history, scan_history_phase0], ignore_index=True, sort=False)
    phase0_summary = summarize_phase0_runs(phase0_history)
    display(phase0_summary.tail(20))
